# VeloGuard - Collecte BOAMP (v7 — description complète via champ donnees)

## Nouveauté v7

Le champ `donnees` de l'API contient le JSON structuré complet de chaque avis,
incluant la description complète sous `donnees > FNSimple > initial > natureMarche > description`.

On n'a **pas besoin de scraper** boamp.fr — tout est dans l'API.
Il suffit d'ajouter `donnees` au SELECT de la collecte.

**Exemple** : le marché 26-34442 a pour titre *'Aménagement de la rue du Charreconduit'*
mais sa description complète précise *'création d'une piste cyclable pour relier
celles existantes à chaque extrémité de la rue'* — détectable uniquement via `donnees`.

---

## 0. Dependances

In [1]:
import requests, pandas as pd, json, time, re, ast
from datetime import datetime, timedelta
from collections import Counter
print('OK')

OK


## 1. Configuration

In [2]:
BASE_URL = "https://boamp-datadila.opendatasoft.com/api/explore/v2.1/catalog/datasets/boamp/records"
JOURS_ARRIERE = 30
MAX_RESULTATS = 5000
OUTPUT_FILE = f"boamp_voirie_{datetime.now().strftime('%Y%m%d')}.csv"

# ── KEYWORDS CYCLABLE ────────────────────────────────────────
KEYWORDS_CYCLABLE = [
    r'piste cyclable', r'bande cyclable', r'voie cyclable',
    r'itin[eé]raire cyclable', r'r[eé]seau cyclable',
    r'v[eé]loroute', r'voie verte', r'couloir v[eé]lo',
    r'am[eé]nagement cyclable', r'continuit[eé] cyclable',
    r'arceaux? v[eé]lo', r'stationnement v[eé]lo',
    r'abri v[eé]lo', r'box v[eé]lo', r'parking v[eé]lo',
    r'borne de recharge.{0,20}v[eé]lo',
    r'v[eé]lo\b', r'v[eé]los\b', r'cycliste',
    r'deux[-\s]roues', r'mobilit[eé] douce',
    r'cheminement doux', r'mode doux',
    r'usager.{0,15}v[eé]lo', r'L\.?228[-\s]2',
]

# ── SCORING PÉRIMÈTRE L228-2 (v5) ────────────────────────────
DESC_VOIRIE_FORT   = ['voirie', 'voirie et réseaux divers', 'chaussée']
DESC_VOIRIE_FAIBLE = ['trottoir', 'revêtement', 'terrassement']

MOTS_REFECTION = [
    r'r[eé]fection.{0,30}(voirie|chauss[eé]e|trottoir|rue|avenue|boulevard)',
    r'r[eé]habilitation.{0,30}(voirie|chauss[eé]e|rue|avenue)',
    r'r[eé]am[eé]nagement.{0,30}(voirie|rue|avenue|boulevard|place|giratoire)',
    r'am[eé]nagement.{0,30}(voirie|rue|avenue|boulevard|place|giratoire|carrefour)',
    r'requalification.{0,30}(voirie|rue|avenue|boulevard|place)',
    r'travaux.{0,20}voirie', r'entretien.{0,20}voirie',
    r'am[eé]nagement urbain', r'enrob[eé]', r'enduit superficiel',
    r'rev[eê]tement.{0,20}(chauss[eé]e|voirie|rue)',
    r'trottoir', r'chauss[eé]e', r'giratoire',
    r'carrefour.{0,20}(am[eé]nag|s[eé]curis)',
]

EXCLUSIONS_FORTES = [
    r'\b(b[aâ]timent|r[eé]novation.{0,20}b[aâ]t|construction.{0,20}b[aâ]t)\b',
    r'\b(r[eé]seau.{0,15}(ftth|fibre|eau potable|gaz|assainissement))\b',
    r'\b(toiture|charpente|menuiserie|peinture.{0,10}b[aâ]t|plomberie)\b',
    r'\b(barrage|berge|digue)\b',
    r'\b(ascenseur|escalier)\b',
    r'\bd[eé]samiantage\b',
    r'\b(pharmacie|h[oô]pital|chru?|ehpad)\b',
    r'\bstade\b', r'\btribune\b',
    r'\bcimeti[eè]re\b', r'\ba[eé]roport\b',
]

HORS_PERIMETRE_INFRA = [
    r'\bRN\s*\d+\b', r'\bautoroute\b',
    r'\bRD\s*\d+.{0,40}(hors agglo|route d[eé]partementale.{0,20}(entre|pr\s*\d|section))',
]

print("OK - configuration chargee")
print(f"   {len(KEYWORDS_CYCLABLE)} patterns cyclable")
print(f"   {len(MOTS_REFECTION)} mots refection | {len(EXCLUSIONS_FORTES)} exclusions fortes")

OK - configuration chargee
   25 patterns cyclable
   15 mots refection | 11 exclusions fortes


## 2. Fonctions de scoring

In [6]:
def parse_descripteurs(val):
    try:
        lst = ast.literal_eval(str(val))
        if isinstance(lst, list):
            return [str(x).lower().strip() for x in lst]
    except:
        pass
    return [s.strip().lower() for s in str(val).split(',')]

def score_perimetre_l228(row):
    descs  = parse_descripteurs(row.get('descripteur_libelle', row.get('descripteur_str', '')))
    objet  = str(row.get('objet', '')).lower()
    nb_desc = len(descs)
    score  = 0

    has_voirie_fort = any(any(v in d for v in DESC_VOIRIE_FORT) for d in descs)
    if has_voirie_fort:
        if nb_desc > 8:
            score += 0   # voirie = lot mineur dans accord-cadre TCE
        elif nb_desc > 5:
            score += 1
        else:
            score += 2
    elif any(any(v in d for v in DESC_VOIRIE_FAIBLE) for d in descs):
        if nb_desc <= 4:
            score += 1

    for p in MOTS_REFECTION:
        if re.search(p, objet, re.IGNORECASE):
            score += 1
            break

    for p in EXCLUSIONS_FORTES:
        if re.search(p, objet, re.IGNORECASE):
            score -= 3
            break

    for p in HORS_PERIMETRE_INFRA:
        if re.search(p, objet, re.IGNORECASE):
            score -= 2
            break

    if re.search(r'\bconstruction\b.{0,30}\b(neuve|b[aâ]timent|logements|maisons?)\b', objet, re.IGNORECASE):
        score -= 1

    return score

def detecter_cyclable(texte):
    t = str(texte).lower()
    mots = []
    for p in KEYWORDS_CYCLABLE:
        m = re.findall(p, t, re.IGNORECASE)
        if m:
            mots.extend([str(x).lower() for x in m])
    return bool(mots), list(set(mots))

print("OK - fonctions scoring definies")

OK - fonctions scoring definies


## 3. Collecte BOAMP

In [7]:
def fetch_page(where, q=None, offset=0, limit=100):
    params = {
        'where': where, 'order_by': 'dateparution DESC',
        'limit': limit, 'offset': offset,
        'lang': 'fr', 'timezone': 'Europe/Paris',
    }
    if q:
        params['q'] = q
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_all(jours, max_res=5000):
    date_debut = (datetime.now() - timedelta(days=jours)).strftime('%Y-%m-%d')
    where = f'dateparution >= "{date_debut}" AND type_marche:"TRAVAUX"'
    q = '"voirie" OR "chaussee" OR "trottoir" OR "revetement" OR "refection" OR "reamenagement"'
    print(f"Collecte BOAMP — {jours} jours | {where[:50]}...")
    all_records, offset = [], 0
    while offset < max_res:
        try:
            data = fetch_page(where, q=q, offset=offset)
        except requests.HTTPError as e:
            print(f"Erreur {e.response.status_code}: {e.response.text[:200]}")
            break
        records = data.get('results', [])
        total   = data.get('total_count', 0)
        if offset == 0:
            print(f"   -> {total:,} annonces trouvees")
        if not records:
            break
        all_records.extend(records)
        print(f"   -> Page {offset//100+1}: {len(records)} ({len(all_records)}/{min(total,max_res)})")
        if len(all_records) >= total:
            break
        offset += 100
        time.sleep(0.3)
    print(f"OK - {len(all_records)} annonces")
    return all_records

records = fetch_all(JOURS_ARRIERE, MAX_RESULTATS)
df_raw  = pd.DataFrame(records)
print(f"DataFrame: {df_raw.shape}")

Collecte BOAMP — 30 jours | dateparution >= "2026-05-02" AND type_marche:"TRAV...
   -> 2,386 annonces trouvees
   -> Page 1: 100 (100/2386)
   -> Page 2: 100 (200/2386)
   -> Page 3: 100 (300/2386)
   -> Page 4: 100 (400/2386)
   -> Page 5: 100 (500/2386)
   -> Page 6: 100 (600/2386)
   -> Page 7: 100 (700/2386)
   -> Page 8: 100 (800/2386)
   -> Page 9: 100 (900/2386)
   -> Page 10: 100 (1000/2386)
   -> Page 11: 100 (1100/2386)
   -> Page 12: 100 (1200/2386)
   -> Page 13: 100 (1300/2386)
   -> Page 14: 100 (1400/2386)
   -> Page 15: 100 (1500/2386)
   -> Page 16: 100 (1600/2386)
   -> Page 17: 100 (1700/2386)
   -> Page 18: 100 (1800/2386)
   -> Page 19: 100 (1900/2386)
   -> Page 20: 100 (2000/2386)
   -> Page 21: 100 (2100/2386)
   -> Page 22: 100 (2200/2386)
   -> Page 23: 100 (2300/2386)
   -> Page 24: 86 (2386/2386)
OK - 2386 annonces
DataFrame: (2386, 41)


## 4. Nettoyage

In [8]:
def nettoyer(df):
    if df.empty:
        return df
    df = df.copy()
    if 'dateparution' in df.columns:
        df['dateparution'] = pd.to_datetime(df['dateparution'], errors='coerce')
    def parse_dept(v):
        try:
            lst = ast.literal_eval(str(v))
            return str(lst[0]).zfill(2) if lst else '00'
        except:
            return str(v).zfill(2)[:2]
    df['dept'] = df.get('code_departement', pd.Series(['00']*len(df))).apply(parse_dept)
    if 'descripteur_libelle' in df.columns:
        df['descripteur_str'] = df['descripteur_libelle'].apply(
            lambda v: ', '.join(parse_descripteurs(v)))
    for col in ['objet', 'nomacheteur']:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str).str.strip()
    return df

df = nettoyer(df_raw)
print("OK")
cols = [c for c in ['idweb','dateparution','dept','nomacheteur','objet'] if c in df.columns]
print(df[cols].head(3).to_string(index=False))

OK
   idweb dateparution dept              nomacheteur                                                                                                  objet
26-53304   2026-06-01   59    COMMUNE DE GRAVELINES MODERNISATION DU PÔLE BASKET "SPORTICA NOUVELLE GÉNÉRATION" - LOT 16 MACHINERIE ET SERRURERIE SCÈNIQUE
26-53325   2026-06-01   73 Département de la Savoie                          Travaux De Construction Du Centre Routier Departemental (Crd) De Voglans (73)
26-53397   2026-06-01   57           Mairie de Yutz                                                                                    Travaux de peinture


## 5. Scoring

In [9]:
if not df.empty:
    df['texte'] = df.get('objet','').fillna('') + ' ' + df.get('descripteur_str','').fillna('')
    df['score_perimetre'] = df.apply(score_perimetre_l228, axis=1)
    df['dans_perimetre']  = df['score_perimetre'] >= 2

    results = df['texte'].apply(detecter_cyclable)
    df['cyclable_detecte'] = results.apply(lambda x: x[0])
    df['cyclable_mots']    = results.apply(lambda x: ', '.join(x[1]) if x[1] else '')
    df['alerte_l228']      = df['dans_perimetre'] & ~df['cyclable_detecte']

    n = len(df)
    np = int(df['dans_perimetre'].sum())
    na = int(df['alerte_l228'].sum())
    nc = np - na

    print(f"{'='*55}")
    print(f"BILAN — {n} annonces TRAVAUX analysees (v5)")
    print(f"{'='*55}")
    print(f"Dans perimetre L228-2 (score >= 2) : {np}")
    print(f"  ALERTES (sans mention cyclable)  : {na}  ({na/np*100:.0f}%)")
    print(f"  Conformes (avec mention cyclable): {nc}  ({nc/np*100:.0f}%)")
    print(f"Hors perimetre                     : {n-np}")
    print()

    cpt = Counter()
    for mots in df[df['cyclable_detecte']]['cyclable_mots']:
        for m in str(mots).split(', '):
            if m.strip(): cpt[m.strip()] += 1
    print("Mots cyclables detectes :")
    for mot, nb_ in cpt.most_common(15):
        print(f"  {nb_:4d}  {mot}")

BILAN — 2386 annonces TRAVAUX analysees (v5)
Dans perimetre L228-2 (score >= 2) : 307
  ALERTES (sans mention cyclable)  : 291  (95%)
  Conformes (avec mention cyclable): 16  (5%)
Hors perimetre                     : 2079

Mots cyclables detectes :
    11  voie verte
     4  véloroute
     3  aménagement cyclable
     3  vélo
     1  itinéraire cyclable
     1  cheminement doux
     1  mode doux
     1  amenagement cyclable
     1  veloroute


In [10]:
# ============================================================
# EXTRACTION DE LA DESCRIPTION COMPLÈTE (champ donnees)
# + RECLASSEMENT DES ALERTES
# ============================================================

def extraire_textes_donnees(donnees_raw):
    """
    Extrait tous les textes utiles du JSON 'donnees' d'un avis BOAMP.
    Parcourt récursivement le JSON pour collecter les valeurs textuelles longues.
    """
    if not donnees_raw:
        return ''
    try:
        d = json.loads(donnees_raw) if isinstance(donnees_raw, str) else donnees_raw
    except Exception:
        return str(donnees_raw)[:2000]

    textes = []

    def parcourir(obj, profondeur=0):
        if profondeur > 10:
            return
        if isinstance(obj, dict):
            for k, v in obj.items():
                # Cibler en priorité les champs description/intitule
                if k.lower() in ('description', 'intitule', 'objet', 'complementinfos',
                                  'capacitetech', 'conditions', 'renseignements',
                                  'informcomplementaire'):
                    if isinstance(v, str) and len(v) > 20:
                        textes.append(v)
                elif isinstance(v, (dict, list)):
                    parcourir(v, profondeur + 1)
                elif isinstance(v, str) and len(v) > 50:
                    textes.append(v)
        elif isinstance(obj, list):
            for item in obj:
                parcourir(item, profondeur + 1)

    parcourir(d)
    return ' '.join(textes)


# Appliquer sur tout le DataFrame
if 'donnees' in df.columns:
    print("Extraction des descriptions complètes depuis le champ donnees...")
    df['description_donnees'] = df['donnees'].apply(extraire_textes_donnees)

    # Stats
    n_avec_desc = (df['description_donnees'].str.len() > 50).sum()
    print(f"  Marchés avec description extraite : {n_avec_desc} / {len(df)}")
    print()

    # Texte complet = objet + descripteur + description donnees
    df['texte_complet'] = (
        df.get('objet', pd.Series(['']*len(df))).fillna('') + ' ' +
        df.get('descripteur_str', pd.Series(['']*len(df))).fillna('') + ' ' +
        df['description_donnees'].fillna('')
    )

    # Ré-appliquer la détection cyclable sur le texte complet
    results_complets = df['texte_complet'].apply(detecter_cyclable)
    df['cyclable_detecte_complet'] = results_complets.apply(lambda x: x[0])
    df['cyclable_mots_complet']    = results_complets.apply(lambda x: ', '.join(x[1]) if x[1] else '')

    # Identifier les reclassements
    # = marchés qui étaient en alerte mais où la description révèle du cyclable
    df['source_cyclable'] = ''
    mask_titre = df['cyclable_detecte']
    mask_desc  = ~df['cyclable_detecte'] & df['cyclable_detecte_complet']

    df.loc[mask_titre, 'source_cyclable'] = 'titre'
    df.loc[mask_desc,  'source_cyclable'] = 'description'

    # Mettre à jour les flags
    df['cyclable_detecte'] = df['cyclable_detecte_complet']
    df['cyclable_mots']    = df['cyclable_mots_complet']
    df['alerte_l228']      = df['dans_perimetre'] & ~df['cyclable_detecte']

    # Bilan
    n_reclasses = mask_desc.sum()
    n_alertes   = int(df['alerte_l228'].sum())
    n_conformes = int((df['dans_perimetre'] & df['cyclable_detecte']).sum())

    print(f"{'='*55}")
    print(f"BILAN APRÈS ANALYSE DESCRIPTION COMPLÈTE")
    print(f"{'='*55}")
    print(f"Marchés reclassés (titre ne mentionnait pas vélo,")
    print(f"  mais la description si)          : {n_reclasses}")
    print(f"Alertes L228-2 restantes           : {n_alertes}")
    print(f"Conformes dans le périmètre        : {n_conformes}")
    print()

    if n_reclasses > 0:
        print("Détail des reclassements :")
        reclasses = df[mask_desc][['dept','objet','cyclable_mots']].head(20)
        for _, r in reclasses.iterrows():
            print(f"  [{r['dept']:>3}] {str(r['objet'])[:65]}")
            print(f"        → {r['cyclable_mots']}")
else:
    print("Colonne 'donnees' absente du DataFrame.")
    print("Vérifier que le SELECT de la collecte inclut 'donnees'.")

Extraction des descriptions complètes depuis le champ donnees...
  Marchés avec description extraite : 2372 / 2386

BILAN APRÈS ANALYSE DESCRIPTION COMPLÈTE
Marchés reclassés (titre ne mentionnait pas vélo,
  mais la description si)          : 11
Alertes L228-2 restantes           : 288
Conformes dans le périmètre        : 19

Détail des reclassements :
  [ 74] Rd54 - Réhabilitation du Pont de Verchaix - Communes de Morillon 
        → voie verte
  [ 59] Le contrat porte sur : Travaux de démolition et de construction d
        → vélos
  [ 30] 1ère phase de l'aménagement de la Vv70 Fontanes/Quissac : Fontane
        → voie verte
  [ 76] Travaux de maintenance électrique des infrastructures maritimes e
        → véloroute
  [ 78] Travaux de réaménagement du pôle gare de Maisons-Laffitte - Lot n
        → vélo, vélos
  [ 66] SUPPRESSION DE DISCONTINUITÉS - CRÉATION D'UNE PASSERELLE SUR L'A
        → voie verte, continuité cyclable
  [ 78] Travaux de réaménagement de la place Charles de Ga

## 5b. Extraction description + reclassement

> Analyse le champ `donnees` de l'API pour détecter les aménagements cyclables
> mentionnés dans la description complète mais pas dans le titre.


In [11]:
# Vérification sur le marché exemple 26-34442
exemple = df[df['idweb'] == '26-34442']
if not exemple.empty:
    r = exemple.iloc[0]
    print("=== MARCHÉ 26-34442 (exemple de reclassement) ===")
    print(f"Titre         : {r.get('objet','')}")
    print(f"Cyclable titre: {r.get('source_cyclable','')}")
    print(f"Mots détectés : {r.get('cyclable_mots','')}")
    print(f"Alerte L228-2 : {r.get('alerte_l228','')}")
    print()
    desc = str(r.get('description_donnees',''))[:400]
    print(f"Description (extrait) : {desc}...")
else:
    print("Marché 26-34442 non présent dans cette collecte (normal si hors période).")

Marché 26-34442 non présent dans cette collecte (normal si hors période).


## 6. Vérification sur un exemple


## 6. Verification qualitative

In [12]:
if not df.empty:
    alertes = df[df['alerte_l228']].sort_values('score_perimetre', ascending=False)
    print(f"TOP ALERTES L228-2 ({len(alertes)}) :")
    cols = [c for c in ['dateparution','dept','score_perimetre','nomacheteur','objet','descripteur_str'] if c in df.columns]
    print(alertes[cols].head(20).to_string(index=False))
    print()
    conformes = df[df['dans_perimetre'] & df['cyclable_detecte']]
    print(f"CONFORMES ({len(conformes)}) :")
    for _, r in conformes.iterrows():
        print(f"  [{r.get('dept','')}] {str(r.get('objet',''))[:65]} | {r.get('cyclable_mots','')}")

TOP ALERTES L228-2 (288) :
dateparution dept  score_perimetre                                      nomacheteur                                                                                                                                                                                          objet                                             descripteur_str
  2026-05-04   09                3                            L'agglo Foix-Varilhes                                                                                                     Grosses réparations de chaussée sur voiries communales et communautaires - Année 2026-2030                                                      voirie
  2026-05-05   56                3                                 COMMUNE DE BRECH                                                                                                                                                        Aménagement de la rue Francis Le Hellec                             

## 7. Export CSV

## 7. Export CSV


In [13]:
COLS = [
    'idweb', 'dateparution', 'dept', 'code_departement',
    'nomacheteur', 'objet', 'descripteur_str',
    'famille_libelle', 'procedure_libelle', 'nature_libelle',
    'score_perimetre', 'dans_perimetre',
    'cyclable_detecte', 'cyclable_mots', 'source_cyclable',
    'alerte_l228', 'url_avis'
]
cols_ok = [c for c in COLS if c in df.columns]
df_out  = df[cols_ok].copy()

if 'dateparution' in df_out.columns:
    df_out['dateparution'] = df_out['dateparution'].dt.strftime('%Y-%m-%d')

df_out.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Exporté : {OUTPUT_FILE}  ({len(df_out)} lignes)")
print()
print("Colonne source_cyclable :")
if 'source_cyclable' in df_out.columns:
    print(df_out[df_out['dans_perimetre']]['source_cyclable'].value_counts().to_string())

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    pass

Exporté : boamp_voirie_20260601.csv  (2386 lignes)

Colonne source_cyclable :
source_cyclable
               288
titre           16
description      3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>